# NeuroMotor - Wrist Dataset

In [1]:
%cd ..

/home/willraftery/PhD/projects/github/emg-stream


In [2]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path("data/neuromotor_wrist")

h5_files = list(data_dir.glob("*.hdf5"))
print(f"Found {len(h5_files)} HDF5 files in {data_dir}")

Found 6 HDF5 files in data/neuromotor_wrist


In [3]:
example_file = h5_files[0]
import pandas as pd

print(example_file)

with h5py.File(example_file, 'r') as f:
    print("Keys in the HDF5 file:", list(f.keys()))

    data = f['data']
    print("Data shape:", data.shape)
    print("Data dtype:", data.dtype)
    print("Data keys:", list(data.attrs.keys()))
    print("Data attributes:", {key: data.attrs[key] for key in data.attrs.keys()})

    time = data['time']
    print("Start Time:", time[0])
    print("End Time:", time[-1])
    print("Duration:", time[-1] - time[0])


    stages = pd.read_hdf(example_file, "stages")
    print(stages.head())

    emg = data['emg']
    print("EMG shape:", emg.shape)



data/neuromotor_wrist/wrist_user_002_dataset_000.hdf5
Keys in the HDF5 file: ['data', 'stages']
Data shape: (3290112,)
Data dtype: [('emg', '<f4', (16,)), ('time', '<f8'), ('wrist_angles', '<f8', (2,))]
Data keys: ['task']
Data attributes: {'task': 'wrist'}
Start Time: 1713965756.2016091
End Time: 1713967401.2215176
Duration: 1645.0199084281921
          start           end                            name
0  1.713966e+09  1.713966e+09                  open_loop_task
1  1.713966e+09  1.713966e+09        hold_neutral_wrist_angle
2  1.713966e+09  1.713966e+09         clockwise_wrist_circles
3  1.713966e+09  1.713966e+09  counterclockwise_wrist_circles
4  1.713966e+09  1.713966e+09         flexions_and_extensions
EMG shape: (3290112, 16)


In [4]:

def load_corpus(dataset_dir: Path) -> pd.DataFrame:
    """Load the corpus CSV defining train/val/test segments."""
    corpus_path = dataset_dir / "wrist_corpus.csv"
    if not corpus_path.exists():
        raise FileNotFoundError(f"Corpus CSV not found at {corpus_path}")
    df = pd.read_csv(corpus_path)
    # Normalise split names to match our convention
    df['split'] = df['split'].str.lower().str.strip()
    return df


def timestamps_to_indices(timestamps: np.ndarray, start_time: float,
                           end_time: float) -> tuple:
    """
    Convert absolute timestamps to array indices.

    Args:
        timestamps: (T,) array of absolute timestamps
        start_time: segment start timestamp
        end_time:   segment end timestamp

    Returns:
        (start_idx, end_idx)
    """
    start_idx = np.searchsorted(timestamps, start_time, side='left')
    end_idx   = np.searchsorted(timestamps, end_time,   side='right')
    return int(start_idx), int(end_idx)

In [ ]:
# Open up the corpus CSV to see what kind of data split we have
corpus_df = load_corpus(data_dir)
print(corpus_df.head())

# Get stats. Is the split inter-subject or intra-subject? How many subjects? How many runs per subject?
num_subjects = corpus_df['user_number'].nunique()
print(f"Number of subjects: {num_subjects}")

# Get number of train/val/test runs
split_counts = corpus_df['split'].value_counts()
print("Split counts:")
print(split_counts)

# As percentage
split_percentages = corpus_df['split'].value_counts(normalize=True) * 100
print("Split percentages:")
print(split_percentages)

# Check if the split is inter-subject or intra-subject
# If each subject appears in only one split, it's inter-subject. Otherwise, it's intra-subject.
subject_split_counts = corpus_df.groupby('user_number')['split'].nunique()
print(subject_split_counts)
# Print which splits each subject appears in
print(corpus_df.groupby('user_number')['split'].unique())
if (subject_split_counts > 1).any():
    print("The split is intra-subject (some subjects appear in multiple splits).")
else:
    print("The split is inter-subject (each subject appears in only one split).")



          start           end  split  dataset_number  user_number  \
0  1.712073e+09  1.712073e+09  train               0           95   
1  1.712074e+09  1.712074e+09  train               0           95   
2  1.712074e+09  1.712074e+09  train               0           95   
3  1.712074e+09  1.712074e+09  train               0           95   
4  1.712074e+09  1.712074e+09  train               0           95   

                           dataset  
0  wrist_user_095_dataset_000.hdf5  
1  wrist_user_095_dataset_000.hdf5  
2  wrist_user_095_dataset_000.hdf5  
3  wrist_user_095_dataset_000.hdf5  
4  wrist_user_095_dataset_000.hdf5  
Number of subjects: 100
Split counts:
split
train    2614
val       372
test      364
Name: count, dtype: int64
Split percentages:
split
train    78.029851
val      11.104478
test     10.865672
Name: proportion, dtype: float64
user_number
0     1
1     1
2     1
3     1
4     1
     ..
95    1
96    1
97    1
98    1
99    1
Name: split, Length: 100, dtype: int

In [ ]:
corpus_by_file = load_corpus(data_dir)

# Quick diagnostic — run on one file before full conversion
with h5py.File(h5_files[0], 'r') as src:
    timestamps = np.array(src['data']['time'])
    stages     = pd.read_hdf(h5_files[0], "stages")

fname          = h5_files[0].name
file_segments  = corpus_by_file.get_group(fname)

for _, seg in file_segments.iterrows():
    stage_id, stage_name = get_stage_label(seg["start"], seg["end"], stages)
    duration = seg["end"] - seg["start"]
    print(f"  seg [{seg['start']:.1f} → {seg['end']:.1f}] "
          f"({duration:.1f}s) → stage: {stage_name} (id={stage_id})")

NameError: name 'corpus_by_file' is not defined